# task_11 Solution

In [ ]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_11/data/")


In [ ]:

meters = pd.read_csv(base / "meters.csv")
usage = pd.read_csv(base / "usage.csv")
tariffs = pd.read_csv(base / "tariffs.csv")
outages = pd.read_csv(base / "outages.csv")

Q4: Among meters with at least two outages, which meter_id has the largest drop in average kwh during outage-window hours versus the same clock hours on non-outage dates?

In [ ]:
usage = pd.read_csv(base / "usage.csv", parse_dates=["timestamp"])
outages = pd.read_csv(base / "outages.csv", parse_dates=["outage_start", "outage_end"])

usage["hour"] = usage["timestamp"].dt.hour
usage["date"] = usage["timestamp"].dt.date
eligible_meters = outages.groupby("meter_id").size()
eligible_meters = eligible_meters[eligible_meters >= 2].index

rows = []
for meter_id in eligible_meters:
    meter_usage = usage[usage["meter_id"] == meter_id].copy()
    meter_outages = outages[outages["meter_id"] == meter_id]
    meter_usage["in_outage_window"] = False
    outage_hours = set()
    outage_dates = set()

    for row in meter_outages.itertuples(index=False):
        meter_usage.loc[(meter_usage["timestamp"] >= row.outage_start) & (meter_usage["timestamp"] < row.outage_end), "in_outage_window"] = True
        outage_hours.update(range(row.outage_start.hour, row.outage_end.hour))
        outage_dates.add(row.outage_start.date())

    comparable = meter_usage[meter_usage["hour"].isin(outage_hours)].copy()
    outage_mean = comparable[comparable["in_outage_window"]]["kwh"].mean()
    baseline_mean = comparable[(~comparable["in_outage_window"]) & (~comparable["date"].isin(outage_dates))]["kwh"].mean()
    rows.append({"meter_id": meter_id, "drop": baseline_mean - outage_mean})


drops = pd.DataFrame(rows).sort_values(["drop", "meter_id"], ascending=[False, True])
q4 = str(drops.iloc[0]["meter_id"])
print(q4)

MTR05
